# Infant Formula Comparison: Save web pages to PDFs

This notebook copies all the web pages for the formulas listed into a folder so that we can see the packaging and the ingredient list.
All path resolution delegates to the installed `infant_formula` package — no hardcoded paths in notebook cells.

In [43]:
import time
import random
import pandas as pd
from playwright.async_api import async_playwright
from infant_formula.save_page import CSV_PATH, WEB_LINKS_DIR

In [44]:
df = pd.read_csv(CSV_PATH, sep='\t')

In [ ]:
CLICK_THROUGH_LABELS = [
    "Reject All",                                # cookie first — overlay blocks everything else
    "Deny Non-Essential",
    "Reject All Cookies",
    "Accept All Cookies",                        # cookie — last resort
    "Accept",
    "I confirm I am a Healthcare Professional",  # HCP gate — after cookie overlay is gone
    "I confirm I am a Patient/Carer",            # HCP gate — fallback
]

async def _try_click(page, label: str) -> bool:
    for role in ("button", "link"):
        try:
            await page.get_by_role(role, name=label).first.click(timeout=2_000)
            return True
        except:
            continue
    return False

async def save_pdf(url: str, brand: str, model: str, index_100: int) -> None:
    WEB_LINKS_DIR.mkdir(parents=True, exist_ok=True)
    pdf_path = WEB_LINKS_DIR / f"{index_100}_{brand}_{model}.pdf"
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page()
        await page.emulate_media(media="screen")
        await page.goto(url, wait_until="load", timeout=60_000)
        await page.wait_for_load_state("networkidle", timeout=15_000)
        for label in CLICK_THROUGH_LABELS:
            clicked = await _try_click(page, label)
            if clicked:
                await page.wait_for_load_state("networkidle", timeout=10_000)
        await page.pdf(path=str(pdf_path), print_background=True)
        await browser.close()
    print(f"Saved: {pdf_path}")

In [49]:
# Save one page — test before running the full loop
row = df[df['index_100'] == 101].iloc[0]
await save_pdf(row['website_link'], row['brand'], row['model'], 101)

TimeoutError: Timeout 15000ms exceeded.

In [47]:
display(df.head(4))

,index_100,brand,model,website_link,ingredient_list,has_soy,has_palm_oil,total_arsenic,inorganic_arsenic,lead,cadmium,mercury,aluminum,potassium,bisphenol_a,bisphenol_f,bisphenol_s,acrylamide,report_year
0,101,A2,Platinum,https://a2platinum.com/pages/why-a2,recalled,NaN,NaN,ND,NT,2.9,ND,ND,1770,5580000,ND,ND,ND,ND,2025
1,102,Aptamil,First Infant Milk,https://www.nutricia.co.uk/hcp/pim-products/ap...,Dairy-based blend (of which 29% is fermented) ...,NaN,NaN,ND,NT,1.5,ND,ND,215,5670000,ND,ND,ND,ND,2025
2,103,Baby’s Only\nOrganic,Complete Nutrition,https://babysonly.com/collections/shop-all/pro...,"Organic Lactose, Organic Nonfat Milk (Source O...",NaN,NaN,ND,NT,1.4,ND,ND,307,6080000,ND,ND,ND,ND,2025
3,104,Bobbie,Grass-Fed Whole Milk\n(non-organic),https://www.hibobbie.com/products/grass-fed-wh...,"lactose, organic whole milk, organic linoleic ...",NaN,NaN,ND,NT,ND,ND,ND,95,7080000,ND,ND,ND,ND,2026


In [50]:
for idx in df['index_100'].tolist():
    row = df[df['index_100'] == idx].iloc[0]
    if pd.isna(row['website_link']) or not str(row['website_link']).strip():
        print(f"Skipping {idx} — no URL")
        continue
    await save_pdf(row['website_link'], row['brand'], row['model'], idx)
    time.sleep(random.uniform(2, 4))

TimeoutError: Timeout 15000ms exceeded.